# MapBiomas Fire — Validação de Export GCS (Peru)

Verifica se os **chunks (M1)** e **COGs (M2)** do monitor Sentinel-2 do Peru foram exportados corretamente para o GCS.

## Estrutura validada

| Campo | Valor |
|-------|-------|
| Bucket | `mapbiomas-fire` |
| Raiz | `sudamerica/peru/CATALOG_01/LIBRARY_IMAGES/SENTINEL2/MONTHLY/{MOSAIC}/{YYYY_MM}/` |
| Métodos | `MINNBR`, `MINNBR_BUFFER` |
| Bandas | `blue`, `green`, `red`, `nir`, `swir1`, `swir2`, `dayOfYear` |
| Sensores | `SENTINEL2` |

**Zero download**: só metadados (contagem, tamanhos).

In [ ]:
# @title 1. Setup — Ambiente e Autenticação
import os, sys, json, datetime, time, re, textwrap, warnings
warnings.filterwarnings('ignore')
from collections import defaultdict

IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ
print(f'Ambiente: {"Google Colab" if IS_COLAB else "Local"}')

if IS_COLAB:
    !pip install -q gcsfs pandas tabulate 2>&1 | tail -1
    from google.colab import auth
    auth.authenticate_user()
    print('GCP autenticado via Colab.')
else:
    print('Assumindo Application Default Credentials (ADC) local.')

import gcsfs
import pandas as pd

fs = gcsfs.GCSFileSystem()
print('gcsfs pronto.')

In [ ]:
# @title 2. Configuração
BUCKET = 'mapbiomas-fire'
GCS_BASE = f'{BUCKET}/sudamerica/peru/CATALOG_01/LIBRARY_IMAGES/SENTINEL2/MONTHLY'
MOSAIC_METHODS = ['MINNBR', 'MINNBR_BUFFER']
BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'dayOfYear']
COUNTRY = 'peru'
SENSOR = 'sentinel2'

# Gera o nome base do mosaico igual ao mosaic_name() do M0
def mosaic_name(year, month, band, mosaic):
    m = mosaic.lower()
    return f'image_{COUNTRY}_fire_{SENSOR}_{m}_{band}_{year}_{month:02d}'

print(f'Bucket: {BUCKET}')
print(f'Base: {GCS_BASE}')
print(f'Métodos: {MOSAIC_METHODS}')
print(f'Bandas: {BANDS}')

In [ ]:
# @title 3. Varredura do GCS
def scan_gcs(mosaic_methods=None, bands=None):
    """Escaneia GCS e retorna lista de dicts com metadados de chunks e COGs."""
    mosaic_methods = mosaic_methods or MOSAIC_METHODS
    bands = bands or BANDS
    rows = []
    total_months = 0
    
    for mosaic in mosaic_methods:
        base = f'{GCS_BASE}/{mosaic}'
        
        # Lista diretórios de mês (YYYY_MM)
        try:
            entries = fs.ls(base)
            months = sorted([
                e.rstrip('/').split('/')[-1]
                for e in entries
                if fs.isdir(e) and re.match(r'^\d{4}_\d{2}$', e.rstrip('/').split('/')[-1])
            ])
        except Exception as e:
            print(f'  [SKIP] {mosaic}: não foi possível listar — {e}')
            continue
        
        total_months += len(months)
        print(f'  [{mosaic}] {len(months)} meses encontrados')
        
        for ym in months:
            # --- CHUNKS ---
            chunk_prefix = f'{base}/{ym}/CHUNKS'
            chunk_files = []
            try:
                all_files = fs.find(chunk_prefix)
                chunk_files = [f for f in all_files if f.endswith('.tif')]
            except FileNotFoundError:
                pass  # Pasta CHUNKS não existe
            except Exception as e:
                print(f'    [WARN] {ym} CHUNKS: {e}')
            
            # --- COGs ---
            cog_prefix = f'{base}/{ym}/COG'
            cog_files = []
            try:
                all_cogs = fs.find(cog_prefix)
                cog_files = [f for f in all_cogs if f.endswith('_cog.tif')]
            except FileNotFoundError:
                pass  # Pasta COG não existe
            except Exception as e:
                print(f'    [WARN] {ym} COG: {e}')
            
            # Agrupa por banda (case-insensitive)
            for band in bands:
                b_lower = band.lower()
                
                # Filtra chunks desta banda
                band_chunks = [
                    f for f in chunk_files
                    if f'_{b_lower}_' in f.lower() or f'_{b_lower}.' in f.lower()
                ]
                
                # Filtra COGs desta banda
                band_cogs = [
                    f for f in cog_files
                    if f'_{b_lower}_' in f.lower() or f'_{b_lower}.' in f.lower()
                ]
                
                # Tamanhos
                sizes = []
                for f in band_chunks:
                    try:
                        sizes.append(fs.info(f)['size'])
                    except:
                        pass
                
                cog_size = 0
                if band_cogs:
                    try:
                        cog_size = fs.info(band_cogs[0])['size']
                    except:
                        pass
                
                rows.append({
                    'mosaic': mosaic,
                    'year_month': ym,
                    'year': int(ym[:4]),
                    'month': int(ym[5:7]),
                    'band': band,
                    'n_chunks': len(band_chunks),
                    'chunks_bytes': sum(sizes),
                    'chunks_min': min(sizes) if sizes else 0,
                    'chunks_max': max(sizes) if sizes else 0,
                    'n_cogs': len(band_cogs),
                    'has_cog': len(band_cogs) > 0,
                    'cog_bytes': cog_size,
                })
    
    return rows, total_months

print('Iniciando varredura do GCS...')
t0 = time.time()
records, n_months = scan_gcs()
elapsed = time.time() - t0
print(f'\nVarredura concluída em {elapsed:.1f}s')
print(f'Total: {len(records)} registros (bandas × meses), {n_months} meses')

In [ ]:
# @title 4. Validação e Tabela Resumo
df = pd.DataFrame(records)

if df.empty:
    print('Nenhum dado encontrado no GCS.')
else:
    # Converte bytes → MB
    df['chunks_mb'] = (df['chunks_bytes'] / 1e6).round(1)
    df['chunks_avg_mb'] = (df['chunks_bytes'] / df['n_chunks'] / 1e6).round(2)
    df['cog_mb'] = (df['cog_bytes'] / 1e6).round(1)
    
    # Regras de validação
    def classify(row):
        if row['n_chunks'] == 0:
            return 'MISSING'
        if row['has_cog']:
            return 'OK'
        return 'CHUNKS_ONLY'  # chunks existem mas COG não foi montado
    
    df['status'] = df.apply(classify, axis=1)
    
    # Paleta de cores
    STATUS_COLORS = {
        'OK': 'background-color: #d4edda; color: #155724',
        'CHUNKS_ONLY': 'background-color: #fff3cd; color: #856404',
        'MISSING': 'background-color: #f8d7da; color: #721c24',
    }
    
    # --- TABELA DETALHADA ---
    display_cols = ['mosaic', 'year_month', 'band', 'n_chunks', 'chunks_mb',
                    'chunks_avg_mb', 'has_cog', 'cog_mb', 'status']
    detail = df[display_cols].copy()
    
    styled = detail.style \
        .applymap(lambda s: STATUS_COLORS.get(s, ''), subset=['status']) \
        .format({
            'chunks_mb': '{:.1f}',
            'chunks_avg_mb': '{:.2f}',
            'cog_mb': '{:.1f}',
        }) \
        .set_caption('Validação Chunks e COGs — Peru Sentinel-2')
    
    display(styled)
    
    # --- RESUMO POR MÊS ---
    print('\n' + '='*80)
    print('RESUMO POR MÊS')
    print('='*80)
    
    for mosaic in df['mosaic'].unique():
        sub = df[df['mosaic'] == mosaic]
        print(f'\n--- {mosaic} ---')
        for ym in sorted(sub['year_month'].unique(), reverse=True):
            row = sub[sub['year_month'] == ym]
            n_ok = (row['status'] == 'OK').sum()
            n_chunks_only = (row['status'] == 'CHUNKS_ONLY').sum()
            n_miss = (row['status'] == 'MISSING').sum()
            n_bands_total = len(row)
            
            if n_ok == n_bands_total:
                icon = '✅'
            elif n_chunks_only > 0:
                icon = '⚠️'
            else:
                icon = '❌'
            
            print(f'  {icon} {ym}: {n_ok}/{n_bands_total} COGs OK'
                  f'{" | " + str(n_chunks_only) + " só chunks" if n_chunks_only else ""}'
                  f'{" | " + str(n_miss) + " faltando" if n_miss else ""}')

In [ ]:
# @title 5. Anomalias
print('='*80)
print('ANOMALIAS DETECTADAS')
print('='*80)

anomalies = []

# --- 1. Chunks sem COG ---
chunks_only = df[df['status'] == 'CHUNKS_ONLY']
for _, row in chunks_only.iterrows():
    msg = f'  ⚠️  [{row["mosaic"]}] {row["year_month"]} / {row["band"]}: {row["n_chunks"]} chunks, mas SEM COG'
    print(msg)
    anomalies.append(msg)

# --- 2. Bandas completamente faltando ---
missing = df[df['status'] == 'MISSING']
for _, row in missing.iterrows():
    msg = f'  ❌ [{row["mosaic"]}] {row["year_month"]} / {row["band"]}: 0 chunks (banda ausente)'
    print(msg)
    anomalies.append(msg)

# --- 3. Chunks com tamanho anômalo (fora de 3× std de meses vizinhos) ---
for mosaic in df['mosaic'].unique():
    for band in BANDS:
        sub = df[(df['mosaic'] == mosaic) & (df['band'] == band)]
        sizes = sub[sub['n_chunks'] > 0]['chunks_avg_mb']
        if len(sizes) >= 3:
            mean = sizes.mean()
            std = sizes.std()
            outliers = sub[(sub['n_chunks'] > 0) &
                          ((sub['chunks_avg_mb'] < mean - 3*std) |
                           (sub['chunks_avg_mb'] > mean + 3*std))]
            for _, row in outliers.iterrows():
                msg = (f'  🔍 [{row["mosaic"]}] {row["year_month"]} / {row["band"]}: '
                       f'tamanho médio {row["chunks_avg_mb"]}MB '
                       f'(média {mean:.2f}MB, std {std:.2f}MB)')
                print(msg)
                anomalies.append(msg)

if not anomalies:
    print('  Nenhuma anomalia detectada. ✅')
else:
    print(f'\n  Total: {len(anomalies)} anomalia(s)')

In [ ]:
# @title 5b. Completude por Ano
print("=" * 80)
print("COMPLETUDE POR ANO")
print("=" * 80)

today = datetime.date.today()
current_year = today.year
current_month = today.month

# Meses distintos (mosaic, year_month) com dados
found = df[["mosaic", "year", "year_month"]].drop_duplicates()

year_rows = []
for mosaic in sorted(found["mosaic"].unique()):
    sub = found[found["mosaic"] == mosaic]
    years = sorted(sub["year"].unique())

    for year in years:
        is_current = (year == current_year)

        # Quantos meses esperar?
        if year < current_year:
            n_expected = 12
        elif is_current:
            # Ano corrente: espera dados ate o mes anterior
            n_expected = max(0, current_month - 1)
        else:
            # Ano futuro
            n_expected = 0

        existing = set(sub[sub["year"] == year]["year_month"])
        expected_months = [f"{year}_{m:02d}" for m in range(1, n_expected + 1)]
        found_list = [ym for ym in expected_months if ym in existing]
        missing_list = [ym for ym in expected_months if ym not in existing]

        if n_expected == 0:
            pct = 100.0
        else:
            pct = len(found_list) / n_expected * 100

        if n_expected == 0:
            icon = chr(0x2795)
            status = "FUTURE"
        elif len(found_list) >= n_expected:
            icon = chr(0x2705)
            status = "COMPLETE"
        elif len(found_list) >= n_expected / 2:
            icon = chr(0x26A0) + chr(0xFE0F)
            status = "PARTIAL"
        else:
            icon = chr(0x274C)
            status = "SPARSE"

        extra = " (Ano corrente)" if is_current else ""
        print(f"  {icon} [{mosaic}] {year}: {len(found_list)}/{n_expected} meses ({pct:.0f}%){extra}")
        if missing_list:
            print(f"       Faltando: {", ".join(missing_list)}")

        year_rows.append({
            'mosaic': mosaic,
            'year': year,
            'year_current': is_current,
            'months_found': len(found_list),
            'months_expected': n_expected,
            'completude_pct': f'{round(pct)}%',
            'status': status,
            'missing_months': ', '.join(missing_list) if missing_list else '-',
        })

df_year = pd.DataFrame(year_rows)
print(f"\nTotal: {len(df_year)} anos x mosaicos analisados")


In [ ]:
# @title 6. Download do Relatório
import base64, textwrap
from IPython.display import display, HTML

if not df.empty:
    timestamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    
    # --- CSV ---
    csv_bytes = df.to_csv(index=False).encode('utf-8')
    csv_b64 = base64.b64encode(csv_bytes).decode()
    csv_name = f'validation_report_{timestamp}.csv'
    
    # --- HTML colorido ---
    _C = {'OK': '#d4edda', 'CHUNKS_ONLY': '#fff3cd', 'MISSING': '#f8d7da'}
    html_table = df.style \
        .applymap(lambda s: f'background-color: {_C.get(s, "")}', subset=['status']) \
        .format({'chunks_bytes': '{:,}', 'chunks_min': '{:,}',
                 'chunks_max': '{:,}', 'cog_bytes': '{:,}'}) \
        .hide(axis='index').to_html()

    # --- Tabela de completude por ano ---
    if not df_year.empty:
        _CY = {'COMPLETE': '#d4edda', 'PARTIAL': '#fff3cd', 'SPARSE': '#f8d7da', 'FUTURE': '#e2e3e5'}
        html_year = df_year.style \
            .applymap(lambda s: f'background-color: {_CY.get(s, "")}', subset=['status']) \
            .hide(axis='index').to_html()
    else:
        html_year = '<p>Sem dados de ano.</p>'

    
    html_content = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Validação GCS — Peru Fire {timestamp}</title>
<style>
body {{ font-family: sans-serif; margin: 20px; }}
table {{ border-collapse: collapse; }}
td, th {{ padding: 4px 8px; border: 1px solid #ccc; font-size: 12px; }}
th {{ background: #f0f0f0; }}
</style></head><body>
<h2>Validação Export GCS — Peru Fire Monitor</h2>
<p>Gerado em: {timestamp}</p>
{html_table}
{html_year}
</body></html>'''
    html_b64 = base64.b64encode(html_content.encode()).decode()
    html_name = f'validation_report_{timestamp}.html'
    
    # Exibe a tabela colorida inline
    display(HTML(f'<h3>Relatório de Validação — {timestamp}</h3>' + html_table))
    if not df_year.empty:
        display(HTML(f'<h4>Completude por Ano</h4>' + html_year))
    
    # Botões de download
    display(HTML(f'''
    <div style="margin: 20px 0;">
      <a download="{csv_name}" href="data:text/csv;base64,{csv_b64}"
         style="padding: 10px 20px; background: #28a745; color: white; text-decoration: none; border-radius: 4px; margin-right: 10px;">
        📥 Download CSV
      </a>
      <a download="{html_name}" href="data:text/html;base64,{html_b64}"
         style="padding: 10px 20px; background: #007bff; color: white; text-decoration: none; border-radius: 4px;">
        📥 Download HTML
      </a>
    </div>
    '''))
else:
    print('Nada a exportar (DataFrame vazio).')

---
**Fim da validação.** 

Se encontrou anomalias, revise os meses/métodos sinalizados e reexecute M1 (export) ou M2 (mosaic) conforme necessário.